# 🏪 FahMai RAG Challenge — Multi-Strategy Router + Voting Pipeline
## Super AI Engineer Season 6 — Level 1

ระบบ RAG สำหรับตอบคำถามเกี่ยวกับร้านฟ้าใหม่ (FahMai Electronics Store)

**สถาปัตยกรรมหลัก:**
- 4-Stage Question Router (NOT_RELATED → Product ID → Type Classification → Strategy)
- Multi-Model Voting (3 โมเดล: typhoon, openthaigpt, kbtg)
- Choice Shuffling เพื่อลด positional bias (3 permutations × 3 models = 9 votes)
- Anti-bias heuristic สำหรับคำตอบที่ 1 และ 3-way splits
- Smart Context Management (14K token budget)
- TF-IDF Embedding Fallback

## 1. ติดตั้งและตั้งค่า

In [ ]:
# ตั้งค่า API Key จาก Colab Secrets
from google.colab import userdata
THAILLM_API_KEY = userdata.get('ThaiLLM')
print(f'API Key loaded: {THAILLM_API_KEY[:8]}...' if THAILLM_API_KEY else 'ERROR: ไม่พบ API Key')

In [ ]:
# อัปโหลดและแตกไฟล์ data.zip
import os
if not os.path.exists('/content/data'):
    !unzip -q data.zip -d /content/
    print('แตกไฟล์ data.zip เรียบร้อย')
else:
    print('มีโฟลเดอร์ data อยู่แล้ว')

# ตั้งค่าโฟลเดอร์
DATA_DIR = '/content/data'
KB_DIR = os.path.join(DATA_DIR, 'knowledge_base')
QUESTIONS_CSV = os.path.join(DATA_DIR, 'questions.csv')

# ตั้งจำนวนคำถาม (10 สำหรับ demo, 100 สำหรับ submission จริง)
N_QUESTIONS = 10  # เปลี่ยนเป็น 100 สำหรับ full run
print(f'จะประมวลผล {N_QUESTIONS} คำถาม')

## 2. โหลดข้อมูล Knowledge Base และคำถาม

In [ ]:
import csv
from pathlib import Path

# โหลด KB files ทั้งหมด
def load_kb_files(kb_dir):
    """โหลดไฟล์ markdown ทั้งหมดจาก knowledge_base/"""
    kb = {}
    for root, _, files in os.walk(kb_dir):
        for fname in sorted(files):
            if fname.endswith('.md'):
                fpath = os.path.join(root, fname)
                rel = os.path.relpath(fpath, kb_dir).replace('\\', '/')
                with open(fpath, 'r', encoding='utf-8') as f:
                    kb[rel] = f.read()
    return kb

# โหลดคำถามจาก CSV
def load_questions(csv_path):
    """โหลด questions.csv"""
    questions = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            q = {'id': int(row['id']), 'question': row['question'], 'choices': {}}
            for i in range(1, 11):
                q['choices'][i] = row[f'choice_{i}']
            questions.append(q)
    return questions

kb_files = load_kb_files(KB_DIR)
questions = load_questions(QUESTIONS_CSV)

# สรุปข้อมูล
folder_counts = {}
for rel_path in kb_files:
    folder = rel_path.split('/')[0] if '/' in rel_path else 'root'
    folder_counts[folder] = folder_counts.get(folder, 0) + 1

print(f'KB files ทั้งหมด: {len(kb_files)}')
for folder, count in sorted(folder_counts.items()):
    print(f'  {folder}: {count} ไฟล์')
print(f'คำถามทั้งหมด: {len(questions)}')

## 3. ThaiLLM API Wrapper

In [ ]:
import time
import requests
import re

BASE_URL = 'http://thaillm.or.th/api'
MODELS = ['typhoon', 'openthaigpt', 'pathumma', 'kbtg']

# Rate limiting
_request_times = []
_request_count = 0

def _rate_limit_wait():
    """จำกัด rate: 5 req/sec, 200 req/min"""
    global _request_times
    now = time.time()
    _request_times = [t for t in _request_times if now - t < 60]
    recent_1s = [t for t in _request_times if now - t < 1.0]
    if len(recent_1s) >= 5:
        wait = 1.0 - (now - recent_1s[0]) + 0.05
        if wait > 0: time.sleep(wait)
    if len(_request_times) >= 195:
        wait = 60.0 - (now - _request_times[0]) + 0.1
        if wait > 0:
            print(f'  [Rate limit] รอ {wait:.1f}s...')
            time.sleep(wait)
    _request_times.append(time.time())

def ask_llm(messages, model='typhoon', max_tokens=1024, temperature=0.1, max_retries=2):
    """เรียก ThaiLLM API"""
    global _request_count
    if model not in MODELS:
        model = 'typhoon'
    url = f'{BASE_URL}/{model}/v1/chat/completions'
    headers = {'Content-Type': 'application/json', 'apikey': THAILLM_API_KEY}
    payload = {'model': '/model', 'messages': messages, 'max_tokens': max_tokens, 'temperature': temperature}

    for attempt in range(max_retries):
        try:
            _rate_limit_wait()
            _request_count += 1
            resp = requests.post(url, headers=headers, json=payload, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                if 'choices' in data and len(data['choices']) > 0:
                    return data['choices'][0]['message']['content']
                return None
            elif resp.status_code == 429:
                time.sleep(min(2 ** attempt, 10))
            elif resp.status_code >= 500:
                time.sleep(min(2 ** attempt, 5))
            else:
                if attempt < max_retries - 1: time.sleep(2)
                else: return None
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            time.sleep(min(2 ** attempt, 5))
        except Exception as e:
            print(f'  [Error] {e}')
            if attempt < max_retries - 1: time.sleep(2)
            else: return None
    return None

print('ThaiLLM API wrapper พร้อมใช้งาน')

## 4. Answer Parser — แยกคำตอบจาก LLM

In [ ]:
def parse_answer(response):
    """แยกเลขคำตอบ (1-10) จากข้อความ LLM"""
    if not response: return None
    # ลบ <think> blocks
    text = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    if not text: text = response

    # Pattern หลัก
    patterns = [
        r'ANSWER\s*[:：]\s*(\d{1,2})', r'answer\s*[:：]\s*(\d{1,2})',
        r'คำตอบ\s*[:：]?\s*(\d{1,2})', r'ตัวเลือกที่\s*(\d{1,2})',
        r'ตัวเลือก\s*[:：]\s*(\d{1,2})', r'ข้อ\s*[:：]?\s*(\d{1,2})',
        r'ข้อที่\s*(\d{1,2})', r'เลือก\s*[:：]?\s*(\d{1,2})',
        r'ตอบ\s*[:：]?\s*(\d{1,2})', r'คือ\s*[:：]?\s*(\d{1,2})\b',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            num = int(m.group(1))
            if 1 <= num <= 10: return num

    # Choice pattern
    for pat in [r'choice[_\s]*(\d{1,2})', r'option[_\s]*(\d{1,2})']:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            num = int(m.group(1))
            if 1 <= num <= 10: return num

    # เลขท้ายบรรทัดสุดท้าย
    last_line = text.strip().split('\n')[-1].strip()
    m = re.search(r'^(\d{1,2})\.?$', last_line)
    if m:
        num = int(m.group(1))
        if 1 <= num <= 10: return num

    # Fallback: เลข 1-10 ตัวแรก
    for n_str in re.findall(r'\b(\d{1,2})\b', text):
        num = int(n_str)
        if 1 <= num <= 10: return num
    return None

print('Answer parser พร้อมใช้งาน')

## 5. Context Manager — จัดการ Token Budget

In [ ]:
# Token Budget Constants
MAX_CONTEXT_TOKENS = 14000
SYSTEM_PROMPT_RESERVE = 500
RESPONSE_RESERVE = 1500

def estimate_tokens(text):
    """ประมาณจำนวน token สำหรับข้อความภาษาไทย (~2 chars/token)"""
    return len(text) // 2 if text else 0

def estimate_messages_tokens(messages):
    """ประมาณ token รวมของ messages"""
    return sum(estimate_tokens(m.get('content', '')) + 10 for m in messages)

# Policy Keyword Mapping
POLICY_FILE_KEYWORDS = {
    'policies/return_policy.md': [
        'คืนสินค้า', 'คืนได้', 'คืนเงิน', 'refund', 'ส่งคืน', 'อยากคืน',
        'คืนเฉพาะ', 'แกะกล่อง', 'ไม่แกะกล่อง', 'คืนได้กี่วัน',
        'Mega Sale', 'เมกะเซลล์', 'คืนของ', 'รับคืน',
    ],
    'policies/warranty_policy.md': [
        'ประกัน', 'เคลม', 'Care+', 'on-site', 'รับประกัน',
        'warranty', 'ซ่อม', 'เปลี่ยน', 'on site', 'ซ่อมถึงบ้าน',
        'เสียหาย', 'ชำรุด', 'service', 'บริการซ่อม',
    ],
    'policies/shipping_policy.md': [
        'จัดส่ง', 'ค่าส่ง', 'shipping', 'ขนส่ง', 'พัสดุ',
        'ส่งแบบด่วน', 'ส่งฟรี', 'ค่าจัดส่ง', 'ส่งไป', 'ระยะเวลาจัดส่ง',
        'กี่วันทำการ', 'วันทำการ', 'เกาะ', 'ต่างจังหวัด',
        'ไม่มีลิฟต์', 'ชั้น', 'อพาร์ทเมนต์',
    ],
    'policies/cancellation_policy.md': [
        'ยกเลิก', 'cancel', 'Pre-order', 'พรีออเดอร์',
        'ยกเลิกคำสั่งซื้อ', 'เปลี่ยนใจ', 'สั่งจอง',
        'สถานะ.*จัดส่ง', 'ยกเลิกได้',
    ],
    'policies/membership_points_policy.md': [
        'สมาชิก', 'พอยท์', 'Points', 'Silver', 'Gold', 'Platinum',
        'แต้ม', 'ระดับ', 'FahMai Points', 'คะแนน', 'ระดับสมาชิก',
        'Points.*ลด', 'ใช้.*Points', 'ได้กี่.*Points',
    ],
}

def select_relevant_policies(question, all_policy_files, kb_files):
    """เลือกเฉพาะ policy file ที่เกี่ยวข้อง"""
    scores = {}
    for policy_path, keywords in POLICY_FILE_KEYWORDS.items():
        if policy_path not in kb_files: continue
        score = sum(1 for kw in keywords if re.search(kw, question, re.IGNORECASE))
        if score > 0: scores[policy_path] = score
    if not scores:
        if any(w in question for w in ['ซื้อมา', 'ซื้อ', 'สั่ง']):
            return ['policies/return_policy.md']
        return ['policies/warranty_policy.md']
    sorted_p = sorted(scores.items(), key=lambda x: -x[1])
    selected = [p[0] for p in sorted_p[:2]]
    if len(sorted_p) >= 2 and sorted_p[0][1] >= 2 * sorted_p[1][1]:
        selected = [sorted_p[0][0]]
    return selected

def truncate_content(content, max_tokens):
    """ตัดเนื้อหาให้พอดี budget — เก็บส่วนสำคัญ (spec table, ราคา)"""
    if estimate_tokens(content) <= max_tokens: return content
    lines = content.split('\n')
    important = []
    skip = False
    for line in lines:
        if line.startswith('#'):
            skip = any(s in line.lower() for s in ['รีวิว', 'ความคิดเห็น', 'บทสรุป', 'สรุป', 'เหมาะกับ', 'จุดเด่น', 'รายละเอียดเพิ่มเติม'])
            if not skip: important.append(line)
            continue
        if skip: continue
        if '|' in line or any(kw in line for kw in ['ราคา', 'Price', '฿', 'บาท', 'ประกัน', 'Care+', 'รหัสสินค้า']):
            important.append(line)
        elif line.strip().startswith('-') or line.strip().startswith('*'):
            important.append(line)
        elif line.strip():
            important.append(line)
    result = '\n'.join(important)
    while estimate_tokens(result) > max_tokens and important:
        important.pop()
        result = '\n'.join(important)
    if estimate_tokens(result) > max_tokens:
        result = result[:max_tokens * 2]
    return result

def budget_check_and_truncate_files(file_contents, budget=None):
    """ตรวจสอบและตัดไฟล์ให้พอดี budget"""
    if budget is None:
        budget = MAX_CONTEXT_TOKENS - SYSTEM_PROMPT_RESERVE - RESPONSE_RESERVE - 500
    total = sum(estimate_tokens(c) for c in file_contents)
    if total <= budget: return file_contents
    per_file = budget // len(file_contents)
    return [truncate_content(c, per_file) for c in file_contents]

def select_relevant_files_for_compare(question, relevant_files, kb_files, max_files=3):
    """เลือกไฟล์สำหรับ PRODUCT_COMPARE ให้พอดี budget"""
    if len(relevant_files) <= 2: return relevant_files
    scored = []
    for f in relevant_files:
        content = kb_files.get(f, '')
        first_line = content.split('\n')[0] if content else ''
        product_name = first_line.lstrip('# ').strip()
        score = 0
        if product_name:
            m = re.search(r'\((.+?)\)', product_name)
            eng_name = m.group(1) if m else product_name
            thai_name = re.sub(r'\s*\(.+?\)', '', product_name).strip()
            if eng_name in question: score += 10
            if thai_name in question: score += 10
            for part in eng_name.split():
                if len(part) > 2 and part in question: score += 2
        scored.append((f, score, estimate_tokens(content)))
    scored.sort(key=lambda x: -x[1])
    selected = []
    remaining = MAX_CONTEXT_TOKENS - SYSTEM_PROMPT_RESERVE - RESPONSE_RESERVE
    for f, score, tokens in scored:
        if tokens <= remaining:
            selected.append(f); remaining -= tokens
            if len(selected) >= max_files: break
        elif len(selected) < 2:
            selected.append(f); remaining -= tokens
    return selected if selected else relevant_files[:2]

def build_policy_context(question, kb_files, product_files=None):
    """สร้าง context สำหรับคำถาม POLICY"""
    all_pp = [k for k in kb_files if k.startswith('policies/')]
    selected = select_relevant_policies(question, all_pp, kb_files)
    policy_contents = [kb_files[p] for p in selected if p in kb_files]
    product_content = None
    if product_files:
        p_tokens = sum(estimate_tokens(c) for c in policy_contents)
        budget = MAX_CONTEXT_TOKENS - p_tokens - SYSTEM_PROMPT_RESERVE - RESPONSE_RESERVE - 500
        if budget > 500 and product_files:
            pf = product_files[0]
            if pf in kb_files:
                product_content = truncate_content(kb_files[pf], budget)
    return policy_contents, product_content

def apply_budget(messages, kb_files, question_type='', relevant_files=None):
    """Safety net: ตัด messages ที่เกิน budget"""
    total = estimate_messages_tokens(messages)
    if total <= MAX_CONTEXT_TOKENS: return messages
    for i, msg in enumerate(messages):
        if msg['role'] == 'user':
            content = msg['content']
            overflow = total - MAX_CONTEXT_TOKENS
            target = estimate_tokens(content) - overflow - 500
            if target < 500: target = 500
            messages = messages.copy()
            messages[i] = {**msg, 'content': truncate_content(content, target)}
            break
    return messages

print('Context manager พร้อมใช้งาน')

## 6. Product Mapper — จับคู่ชื่อสินค้ากับไฟล์ KB

In [ ]:
# Brand mappings
BRAND_MAP = {
    'สายฟ้า': 'SaiFah', 'SaiFah': 'สายฟ้า',
    'ดาวเหนือ': 'DaoNuea', 'DaoNuea': 'ดาวเหนือ',
    'คลื่นเสียง': 'KluenSiang', 'KluenSiang': 'คลื่นเสียง',
    'วงโคจร': 'WongKhoJon', 'WongKhoJon': 'วงโคจร',
    'จุดเชื่อม': 'JudChuam', 'JudChuam': 'จุดเชื่อม',
    'ZenByte': 'เซนไบต์', 'เซนไบต์': 'ZenByte',
    'NovaTech': 'โนวาเทค', 'โนวาเทค': 'NovaTech',
    'PulseGear': 'พัลส์เกียร์', 'พัลส์เกียร์': 'PulseGear',
    'ArcWave': 'อาร์คเวฟ', 'อาร์คเวฟ': 'ArcWave',
}

def parse_product_file(filepath, content):
    """แยกข้อมูลสินค้าจากไฟล์ MD"""
    info = {'filepath': filepath, 'sku': '', 'thai_name': '', 'english_name': '',
            'brand_thai': '', 'brand_english': '', 'category': '', 'aliases': []}
    lines = content.split('\n')
    if lines and lines[0].startswith('#'):
        h1 = lines[0].lstrip('# ').strip()
        m = re.match(r'^(.+?)\s*\((.+?)\)\s*$', h1)
        if m:
            info['thai_name'] = m.group(1).strip()
            info['english_name'] = m.group(2).strip()
        else:
            info['thai_name'] = h1
            info['english_name'] = h1
    for line in lines[1:20]:
        line = line.strip()
        if line.startswith('รหัสสินค้า:'):
            info['sku'] = line.split(':', 1)[1].strip()
        elif line.startswith('แบรนด์:'):
            bt = line.split(':', 1)[1].strip()
            bm = re.match(r'^(.+?)\s*\((.+?)\)', bt)
            if bm:
                info['brand_thai'] = bm.group(1).strip()
                info['brand_english'] = bm.group(2).strip()
            else:
                info['brand_thai'] = bt
        elif line.startswith('หมวดหมู่:'):
            info['category'] = line.split(':', 1)[1].strip()

    # สร้าง aliases
    aliases = set()
    if info['thai_name']: aliases.add(info['thai_name'])
    if info['english_name']: aliases.add(info['english_name'])
    # Model name (ลบ brand prefix)
    thai_model = info['thai_name']
    eng_model = info['english_name']
    for brand in [info['brand_thai'], info['brand_english']]:
        if brand and thai_model.startswith(brand):
            thai_model = thai_model[len(brand):].strip()
        if brand and eng_model.startswith(brand):
            eng_model = eng_model[len(brand):].strip()
    if thai_model and len(thai_model) > 2: aliases.add(thai_model)
    if eng_model and len(eng_model) > 2: aliases.add(eng_model)
    if info['sku']: aliases.add(info['sku'])

    # Special aliases
    eng = info['english_name']
    thai = info['thai_name']
    fp = info['filepath']
    # Phone/Tab prefix removal
    for prefix in ['Phone ', 'โฟน ', 'Tab ', 'แท็บ ']:
        for name in [eng, thai]:
            if prefix in name:
                after = name.split(prefix, 1)[1]
                if after: aliases.add(after)
    # Cross-language brand + model
    if info['brand_thai'] and info['brand_english']:
        eng_m = eng
        for b in [info['brand_english'], info['brand_thai']]:
            if eng_m.startswith(b): eng_m = eng_m[len(b):].strip()
        if eng_m: aliases.add(f"{info['brand_thai']} {eng_m}")

    # Thai transliterations
    thai_eng_map = {
        'เฮดโปร': 'HeadPro', 'เฮดออน': 'HeadOn', 'บัดส์': 'Buds',
        'สตูดิโอโปร': 'StudioPro', 'เกมสตอร์ม': 'GameStorm',
        'ซาวด์บาร์': 'SoundBar', 'บูมบ็อกซ์': 'BoomBox', 'โกมินิ': 'Go Mini',
        'โฮมพอด': 'HomePod', 'แอร์บุ๊ก': 'AirBook', 'สตอร์มบุ๊ก': 'StormBook',
        'โปรบุ๊ก': 'ProBook', 'สตัดดี้บุ๊ก': 'StudyBook', 'ครีเอเตอร์บุ๊ก': 'CreatorBook',
        'เฟล็กซ์บุ๊ก': 'FlexBook', 'ทาวเวอร์': 'Tower', 'มินิพีซี': 'Mini PC',
        'ออลอินวัน': 'All-in-One', 'โฟน': 'Phone', 'แท็บ': 'Tab',
        'ดูโอแพด': 'DuoPad', 'พ็อกเก็ตมินิ': 'PocketMini',
        'คิดเซฟ': 'Kid Safe', 'ซีเนียร์ พลัส': 'Senior Plus', 'รักเก็ด': 'Rugged',
        'ฮับ': 'Hub', 'ด็อค': 'Dock', 'เคส': 'Case', 'ฟิล์ม': 'Film',
        'สาย': 'Cable', 'ชาร์จเจอร์': 'Charger', 'ปากกา': 'Pen',
        'แบนด์': 'Band', 'วอทช์': 'Watch', 'ริง': 'Ring',
        'แบตสำรอง': 'Power Bank', 'ชาร์จแพด': 'ChargePad',
        'โนวาบัดส์': 'NovaBuds', 'สลิมบุ๊ก': 'SlimBook',
        'ซาวด์พิลลาร์': 'SoundPillar',
    }

    # Single-word product names
    for name in ['DuoPad', 'PocketMini', 'Kid Safe', 'Senior Plus', 'Rugged R1',
                 'ดูโอแพด', 'พ็อกเก็ตมินิ', 'คิดเซฟ', 'ซีเนียร์ พลัส', 'รักเก็ด R1']:
        if name.lower() in eng.lower() or name in thai: aliases.add(name)

    # Pen
    if 'Pen' in eng and 'Gen' in eng:
        for a in ['SaiFah Pen Gen 2', 'สายฟ้า Pen Gen 2', 'Pen Gen 2']: aliases.add(a)
    if 'QiPad' in eng: aliases.update(['QiPad', 'QiPad 15'])
    if 'ChargePad' in eng: aliases.update(['ChargePad', 'ChargePad 15W'])
    if 'Keyboard Bundle' in eng: aliases.update(['FlexBook Detach Bundle', 'FlexBook Detach Keyboard Bundle'])
    if 'Gold Limited' in eng: aliases.update(['Buds Z5 Pro Gold Limited', 'Buds Z5 Pro Gold'])
    if 'FahMai Edition' in eng: aliases.update(['HeadOn 300 FahMai Edition'])
    if 'Pack 2' in eng: aliases.update(['Go Mini Pack 2', 'Go Mini แพ็คคู่'])

    # Hub aliases
    if 'Hub' in eng:
        for p in ['7-in-1', '4-in-1']:
            if p in eng: aliases.update([f'Hub {p}', f'Hub USB-C {p}'])

    # Charger wattage aliases
    if any(x in eng + thai for x in ['Charger', 'ชาร์จเจอร์', 'หัวชาร์จ']):
        wm = re.search(r'(\d+)\s*[Ww]', eng + ' ' + thai)
        if wm:
            w = wm.group(1)
            aliases.update([f'หัวชาร์จ {w}W', f'Charger {w}W', f'ชาร์จเจอร์ {w}W'])

    # Dock, Power Bank, Cable, Monitor, SoundPillar aliases
    if 'Dock' in eng:
        if 'Pro' in eng: aliases.add('Dock Pro')
        if 'AirBook' in eng: aliases.update(['Dock AirBook Edition', 'Dock AirBook'])
    if 'Power Bank' in eng or 'แบตสำรอง' in thai:
        mm = re.search(r'(\d+)\s*mAh', eng, re.IGNORECASE)
        if mm:
            mah = mm.group(1)
            aliases.update([f'Power Bank {mah}mAh', f'Power Bank {mah}'])
            if len(mah) >= 5:
                fmt = f'{int(mah):,}'
                aliases.update([f'Power Bank {fmt}', f'แบตสำรอง {fmt}'])
    if 'Cable' in eng:
        if 'USB-C to USB-C' in eng:
            lm = re.search(r'(\d+)m', eng)
            if lm: aliases.add(f"Cable USB-C {lm.group(1)}m")
        if 'Lightning' in eng: aliases.add('Cable USB-C to Lightning')
    if 'SSD' in eng:
        cm = re.search(r'(\d+)\s*TB', eng)
        if cm: aliases.update([f'SSD {cm.group(1)}TB', f'SSD NVMe {cm.group(1)}TB'])
    if 'RAM' in eng:
        cm = re.search(r'(\d+)\s*GB', eng)
        if cm: aliases.update([f'DDR5 RAM {cm.group(1)}GB', f'RAM {cm.group(1)}GB'])
    if 'ProView' in eng: aliases.update(['ProView 27 4K', 'ProView 27'])
    if 'SoundPillar' in eng: aliases.update(['SoundPillar 300', 'ซาวด์พิลลาร์ 300'])

    # Sprint 4: Product variant aliases
    if 'stormbook_g5_2024' in fp:
        aliases.update(['StormBook G5 2024', 'StormBook G5 รุ่นปี 2024', 'G5 รุ่นปี 2024', 'G5 2024'])
    if 'airbook_14_8gb' in fp:
        aliases.update(['AirBook 14 8GB', 'AirBook 14 รุ่น 8GB', 'แอร์บุ๊ก 14 8GB'])
    if 'stormbook_g7' in fp:
        aliases.update(['G7', 'StormBook G7', 'สตอร์มบุ๊ก G7'])
    if 'stormbook_g9' in fp:
        aliases.update(['G9 Titan', 'G9'])

    info['aliases'] = list(aliases)
    return info


class ProductMapper:
    """จับคู่ชื่อสินค้าในคำถาม/ตัวเลือกกับไฟล์ KB"""
    def __init__(self, kb_files):
        self.products = []
        self.alias_to_products = {}
        for rel_path, content in kb_files.items():
            if rel_path.startswith('products/'):
                info = parse_product_file(rel_path, content)
                self.products.append(info)
        alias_list = []
        for prod in self.products:
            for alias in prod['aliases']:
                alias_list.append((alias, prod))
        alias_list.sort(key=lambda x: len(x[0]), reverse=True)  # ยาวสุดก่อน
        self._sorted_aliases = alias_list
        for alias, prod in alias_list:
            if alias not in self.alias_to_products:
                self.alias_to_products[alias] = []
            self.alias_to_products[alias].append(prod)

    def find_products(self, question, choices):
        """ค้นหาสินค้าที่ตรงกับคำถามและตัวเลือก"""
        choices_text = ' '.join(str(v) for v in choices.values())
        combined = question + ' ' + choices_text
        found = []
        matched_text = combined
        for alias, prod in self._sorted_aliases:
            if len(alias) < 3: continue
            if alias in matched_text:
                if not any(f['filepath'] == prod['filepath'] for f in found):
                    source = 'question' if alias in question else 'choice'
                    found.append({'name': alias, 'filepath': prod['filepath'], 'sku': prod['sku'],
                                  'english_name': prod['english_name'], 'thai_name': prod['thai_name'], 'source': source})
                matched_text = matched_text.replace(alias, '___MATCHED___')
        # Sprint 5: Accessory dedup
        found = self._dedup_accessory_files(question, found)
        return found

    def _dedup_accessory_files(self, question, found):
        """Sprint 5: ลบไฟล์อุปกรณ์เสริมเมื่อคำถามถามเกี่ยวกับอุปกรณ์หลัก"""
        if len(found) < 2: return found
        device_prefixes = ('SF-SP', 'SF-TB', 'DN-LT', 'NT-LT', 'WK-SW')
        accessory_prefixes = ('JC-CH', 'JC-CB', 'JC-CS', 'JC-FM', 'JC-HB', 'JC-DK', 'PG-PB', 'PG-CP')
        q_devices = [f for f in found if f['source'] == 'question' and
                     any(f['filepath'].split('/')[-1].startswith(p) for p in device_prefixes)]
        q_acc = [f for f in found if f['source'] == 'question' and
                 any(f['filepath'].split('/')[-1].startswith(p) for p in accessory_prefixes)]
        if not q_devices or not q_acc: return found
        signals = ['ซื้อ', 'มาในกล่อง', 'ในกล่อง', 'แถม', 'มากับ', 'มาด้วย',
                   'สเปค', 'คุณสมบัติ', 'รายละเอียด', 'อยากทราบ', 'ได้อะไรบ้าง']
        if any(s in question for s in signals):
            acc_paths = {f['filepath'] for f in q_acc}
            return [f for f in found if f['filepath'] not in acc_paths]
        return found

# สร้าง mapper
mapper = ProductMapper(kb_files)
print(f'Products: {len(mapper.products)}, Aliases: {len(mapper._sorted_aliases)}')

## 7. TF-IDF Embedding Fallback

In [ ]:
import math
from collections import Counter

def tokenize_thai(text):
    """Tokenize ภาษาไทยด้วย character trigrams + words"""
    tokens = []
    for word in text.split():
        tokens.append(word)
        if len(word) >= 3:
            for i in range(len(word) - 2):
                tokens.append(word[i:i+3])
    return tokens

class SimpleTFIDF:
    def __init__(self, documents):
        self.doc_keys = list(documents.keys())
        self.doc_tokens = {}
        self.df = Counter()
        for key, content in documents.items():
            tokens = tokenize_thai(content.lower())
            self.doc_tokens[key] = Counter(tokens)
            for t in set(tokens): self.df[t] += 1
        self.n_docs = len(self.doc_keys)

    def search(self, query, top_k=3):
        query_tf = Counter(tokenize_thai(query.lower()))
        scores = {}
        for key in self.doc_keys:
            score = 0.0
            doc_tf = self.doc_tokens[key]
            for token, qtf in query_tf.items():
                if token in doc_tf:
                    tf = doc_tf[token]
                    idf = math.log((self.n_docs + 1) / (self.df.get(token, 0) + 1))
                    score += tf * idf * qtf
            if score > 0: scores[key] = score
        return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

# สร้าง TF-IDF index
product_files = {k: v for k, v in kb_files.items() if k.startswith('products/')}
tfidf_index = SimpleTFIDF(product_files)

def find_relevant_files_tfidf(question, top_k=3, kb_files=None):
    results = tfidf_index.search(question, top_k=top_k)
    return [path for path, score in results]

print(f'TF-IDF index พร้อม: {tfidf_index.n_docs} documents')

## 8. Product Summarizer — สรุปสินค้าแบบตาราง

In [ ]:
# Category patterns สำหรับคำถามงบ
CATEGORY_PATTERNS = {
    'headphone_over_ear': {'patterns': [r'หูฟังครอบหู', r'หูฟัง.*over.ear'], 'sku_prefix': ['KS-HP'],
        'key_fields': ['ราคา', 'ANC', 'Codec', 'แบตเตอรี่', 'Hi-Res']},
    'headphone_tws': {'patterns': [r'หูฟัง\s*TWS', r'หูฟัง.*ไร้สาย', r'true.wireless', r'earbuds'],
        'sku_prefix': ['KS-EB', 'NT-EB'], 'key_fields': ['ราคา', 'ANC', 'Codec', 'แบตเตอรี่', 'Hi-Res']},
    'headphone_all': {'patterns': [r'หูฟัง(?!.*ครอบหู)(?!.*TWS)'], 'sku_prefix': ['KS-HP', 'KS-EB', 'NT-EB'],
        'key_fields': ['ราคา', 'ประเภท', 'ANC', 'Codec', 'แบตเตอรี่']},
    'speaker': {'patterns': [r'ลำโพง', r'speaker', r'soundbar'], 'sku_prefix': ['KS-SK', 'AW-SK'],
        'key_fields': ['ราคา', 'ประเภท', 'กำลังวัตต์', 'Bluetooth', 'กันน้ำ']},
    'smartwatch': {'patterns': [r'สมาร์ทวอทช์', r'smartwatch'], 'sku_prefix': ['WK-SW'],
        'key_fields': ['ราคา', 'ECG', 'NFC', 'GPS', 'กันน้ำ', 'แบตเตอรี่']},
    'laptop': {'patterns': [r'โน้ตบุ๊ค', r'แล็ปท็อป', r'laptop'], 'sku_prefix': ['DN-LT', 'NT-LT'],
        'key_fields': ['ราคา', 'CPU', 'RAM', 'SSD', 'จอ', 'น้ำหนัก']},
}

def extract_product_summary(filepath, content):
    """สกัดข้อมูลสำคัญจากไฟล์สินค้า"""
    s = {'filepath': filepath, 'name': '', 'price': '', 'specs': {}}
    lines = content.split('\n')
    if lines and lines[0].startswith('#'): s['name'] = lines[0].lstrip('# ').strip()
    pm = re.search(r'ราคา:\s*(฿[\d,]+)', content)
    if pm:
        s['price'] = pm.group(1)
        s['price_num'] = int(pm.group(1).replace('฿','').replace(',',''))
    specs = s['specs']
    specs['ANC'] = '✓' if any(x in content for x in ['ANC', 'Active Noise', 'ตัดเสียงรบกวนเชิงรุก']) else '✗'
    specs['LDAC'] = '✓' if 'LDAC' in content else '✗'
    specs['Hi-Res'] = '✓' if 'Hi-Res' in content else '✗'
    bm = re.search(r'(?:แบตเตอรี่|ใช้งาน|ฟังเพลง).*?(\d+)\s*ชั่วโมง', content)
    if bm: specs['แบตเตอรี่'] = f'{bm.group(1)}ชม'
    specs['Qi'] = '✓' if any(x in content for x in ['Qi', 'ชาร์จไร้สาย']) else '✗'
    specs['ECG'] = '✓' if any(x in content for x in ['ECG', 'คลื่นไฟฟ้าหัวใจ']) else '✗'
    specs['NFC'] = '✓' if 'NFC' in content else '✗'
    specs['GPS'] = '✓' if 'GPS' in content else '✗'
    wm = re.search(r'(IP\d+|ATM|กันน้ำ)', content)
    specs['กันน้ำ'] = wm.group(1) if wm else '✗'
    if 'ครอบหู' in content or 'Over-Ear' in content: specs['ประเภท'] = 'ครอบหู'
    elif 'TWS' in content or 'True Wireless' in content: specs['ประเภท'] = 'TWS'
    codecs = [c for c in ['LDAC', 'aptX', 'AAC', 'SBC'] if c in content]
    if codecs: specs['Codec'] = '/'.join(codecs)
    return s

def detect_question_category(question):
    for cat_name, cat_info in CATEGORY_PATTERNS.items():
        for pattern in cat_info['patterns']:
            if re.search(pattern, question, re.IGNORECASE): return cat_name
    return None

def build_summary_table(question, kb_files, relevant_files=None, max_price=None):
    """สร้างตารางสรุปสินค้าสำหรับคำถามงบ"""
    category = detect_question_category(question)
    if not category: return None
    cat_info = CATEGORY_PATTERNS[category]
    key_fields = cat_info['key_fields']
    if max_price is None:
        pm = re.search(r'(?:งบ|ไม่เกิน|ราคา.*ไม่เกิน)\s*(?:฿)?(\d[\d,]*)', question)
        if pm: max_price = int(pm.group(1).replace(',', ''))
    summaries = []
    files_to_scan = relevant_files if relevant_files else list(kb_files.keys())
    for fp in files_to_scan:
        if not fp.startswith('products/'): continue
        content = kb_files.get(fp, '')
        if not content: continue
        fname = fp.split('/')[-1]
        sku_match = any(fname.startswith(p) for p in cat_info['sku_prefix'])
        if relevant_files or sku_match:
            summaries.append(extract_product_summary(fp, content))
    if not summaries: return None
    summaries.sort(key=lambda s: s.get('price_num', 999999))
    header = '| ชื่อสินค้า | ราคา |'
    sep = '|---|---|'
    for field in key_fields:
        if field != 'ราคา': header += f' {field} |'; sep += '---|'
    rows = []
    for s in summaries:
        row = f"| {s['name'][:40]} | {s['price']} |"
        for field in key_fields:
            if field != 'ราคา': row += f" {s['specs'].get(field, '-')} |"
        rows.append(row)
    table = f'สินค้าในหมวด ({category})'
    if max_price: table += f' — งบไม่เกิน ฿{max_price:,}'
    table += ':\n\n' + header + '\n' + sep + '\n' + '\n'.join(rows)
    return table

print('Product summarizer พร้อมใช้งาน')

## 9. Question Classifier — 4-Stage Router

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class RouteResult:
    question_type: str  # PRODUCT_SPEC, PRODUCT_COMPARE, POLICY, STORE_INFO, NOT_IN_KB, NOT_RELATED, UNKNOWN
    strategy: int       # 0=hardcoded, 1=direct lookup, 3=full file LLM
    relevant_files: List[str] = field(default_factory=list)
    confidence: str = 'MEDIUM'
    matched_products: List[dict] = field(default_factory=list)
    hardcoded_answer: Optional[int] = None

# Keywords สำหรับตรวจจับ
FAHMAI_KEYWORDS = [
    'ฟ้าใหม่', 'FahMai', 'สายฟ้า', 'SaiFah', 'ดาวเหนือ', 'DaoNuea',
    'คลื่นเสียง', 'KluenSiang', 'วงโคจร', 'WongKhoJon', 'จุดเชื่อม', 'JudChuam',
    'ZenByte', 'NovaTech', 'PulseGear', 'ArcWave',
    'เซนไบต์', 'โนวาเทค', 'พัลส์เกียร์', 'อาร์คเวฟ',
    'AirBook', 'StormBook', 'ProBook', 'StudyBook', 'CreatorBook', 'FlexBook',
    'HeadPro', 'HeadOn', 'SoundBar', 'BoomBox', 'HomePod', 'StudioPro', 'GameStorm',
    'Watch S3', 'Band 8', 'Band 7', 'Ring 1', 'Sport G1',
    'X9', 'V7', 'V5', 'Lite 5', 'Lite 3', 'DuoPad', 'PocketMini',
    'Senior Plus', 'Rugged R1', 'Kid Safe',
    'Tab S9', 'Tab A5', 'Tab Mini', 'Tab Kid', 'Tab Draw',
    'Tower X10', 'Mini PC', 'All-in-One',
    'Buds Z5', 'Buds Z3', 'Buds Z1', 'Buds Sport', 'Buds Open Ring',
    'NovaBuds', 'SlimBook',
    'Power Bank', 'ChargePad', 'QiPad',
    'Hub', 'Dock', 'ProView', 'SoundPillar', 'Go Mini',
    'Care+', 'สมาชิก', 'ประกัน', 'คืนสินค้า', 'จัดส่ง',
    'สาขา', 'สั่งซื้อ', 'ออนไลน์', 'Mega Sale',
    'พอยท์', 'Points', 'คะแนน',
    'แอร์บุ๊ก', 'สตอร์มบุ๊ก', 'โปรบุ๊ก', 'ครีเอเตอร์บุ๊ก', 'เฟล็กซ์บุ๊ก',
    'เฮดโปร', 'เฮดออน', 'บัดส์', 'ซาวด์บาร์', 'บูมบ็อกซ์',
    'แบตสำรอง', 'ชาร์จเจอร์', 'หัวชาร์จ',
    'ดูโอแพด', 'พ็อกเก็ตมินิ', 'ซีเนียร์', 'รักเก็ด',
    'SaiFah Pen', 'ปากกา',
    'แท็บ ', 'โฟน ', 'มือถือ',
    'สเปค', 'ราคา', 'กล้อง', 'แบตเตอรี่', 'แบต', 'หน้าจอ', 'ชิปเซ็ต',
    'กันน้ำ', 'ชาร์จ', 'RAM', 'ROM', 'SSD',
    'Bluetooth', 'WiFi', '5G', 'NFC', 'GPS',
    'AMOLED', 'OLED', 'LCD',
    'โน้ตบุ๊ค', 'แล็ปท็อป', 'สมาร์ทโฟน', 'แท็บเล็ต', 'หูฟัง', 'ลำโพง',
    'นาฬิกา', 'สมาร์ทวอทช์',
    'เคส', 'ฟิล์ม', 'Trade-in', 'เทิร์น', 'crypto', 'Cryptocurrency',
]

OFF_TOPIC_SIGNALS = [
    'วันหยุดราชการ', 'ตั๋วเครื่องบิน', 'ดอกเบี้ยเงินฝาก', 'เงินฝากประจำ',
    'สูตรอาหาร', 'ผัดกระเพรา', 'ต้มยำ', 'ส้มตำ',
    'การเมือง', 'เลือกตั้ง', 'ฟุตบอล', 'ฟุตซอล', 'บาสเก็ตบอล',
    'หวย', 'ลอตเตอรี่', 'ดารา', 'ซีรี่ส์', 'ละคร',
    'กรุ๊ปเลือด', 'ราศี', 'ดวง', 'สมุนไพร', 'ยาแผนโบราณ',
    'วิตามิน', 'อาหารเสริม', 'เครื่องบิน', 'สายการบิน', 'โรงแรม',
    'ทองคำ', 'ราคาทอง', 'ตลาดหุ้น',
]

POLICY_KEYWORDS = [
    'คืนสินค้า', 'คืนได้', 'ส่งคืน', 'อยากคืน', 'คืนเฉพาะ',
    'ยกเลิก', 'ยกเลิกคำสั่งซื้อ',
    'เคลม', 'ประกัน', 'warranty',
    'จัดส่ง', 'ค่าส่ง', 'ค่าจัดส่ง', 'ส่งฟรี', 'shipping', 'ส่งแบบด่วน',
    'สมาชิก', 'พอยท์', 'Points', 'คะแนน', 'ระดับสมาชิก',
    'Care+', 'on-site', 'ซ่อม', 'เปลี่ยน',
    'Mega Sale', 'เมกะเซลล์',
    'Pre-order', 'พรีออเดอร์', 'สั่งจอง',
    'สิทธิ์', 'เงื่อนไข', 'Gold', 'Platinum', 'Silver',
]

STORE_KEYWORDS = [
    'Trade-in', 'เทิร์น', 'สั่งได้กี่ชิ้น', 'สั่งได้กี่รายการ', 'ครั้งละกี่',
    'crypto', 'Cryptocurrency', 'Bitcoin',
    'สาขา', 'ที่อยู่ร้าน', 'เปิดกี่โมง',
    'ช่องทาง', 'ติดต่อ', 'โทรศัพท์', 'Line',
    'วิธีชำระ', 'จ่ายเงิน', 'ผ่อน', 'ประวัติ', 'ก่อตั้ง',
]

NOT_IN_KB_PATTERNS = [
    r'ค่าซ่อม|ค่าเปลี่ยน|ราคาซ่อม', r'ผลิตที่ประเทศ|ประเทศผู้ผลิต|ผลิตที่ไหน|โรงงาน',
    r'รายได้|กำไร|ยอดขาย', r'คะแนนรีวิว|รีวิว.*คะแนน|rating',
    r'ค่า SAR|SAR value', r'screen.to.body|สัดส่วนจอ',
    r'ส่วนแบ่งตลาด|market share',
    r'จอ\s*\d+\s*นิ้ว.*ดาวเหนือ|ดาวเหนือ.*จอ\s*\d+\s*นิ้ว|monitor.*ดาวเหนือ',
]

def _get_category_files(question, existing_files):
    """ดึงไฟล์สินค้าทั้งหมดในหมวดที่ตรงกับคำถาม"""
    products_dir = os.path.join(KB_DIR, 'products')
    cat_map = {
        r'หูฟังครอบหู|headphone.*over': ['KS-HP'],
        r'หูฟัง\s*TWS|หูฟัง.*ไร้สาย|earbuds': ['KS-EB', 'NT-EB'],
        r'หูฟัง': ['KS-HP', 'KS-EB', 'NT-EB'],
        r'ลำโพง|speaker|soundbar': ['KS-SK', 'AW-SK'],
        r'สมาร์ทวอทช์|smartwatch|นาฬิกา': ['WK-SW'],
        r'โน้ตบุ๊ค|แล็ปท็อป|laptop': ['DN-LT', 'NT-LT'],
        r'มือถือ|สมาร์ทโฟน|phone': ['SF-SP'],
        r'แท็บเล็ต|tablet': ['SF-TB'],
    }
    for pattern, prefixes in cat_map.items():
        if re.search(pattern, question, re.IGNORECASE):
            files = [f'products/{fname}' for fname in sorted(os.listdir(products_dir))
                     if any(fname.startswith(p) for p in prefixes)]
            return files if files else existing_files
    return existing_files

def classify_question(question_text, choices, product_mapper):
    """Router หลัก: จำแนกคำถามและเลือก strategy"""
    # Stage 1: NOT_RELATED
    choice_10 = choices.get(10, '')
    if 'ไม่เกี่ยวข้อง' in choice_10 or 'not related' in choice_10.lower():
        has_fahmai = any(kw in question_text for kw in FAHMAI_KEYWORDS)
        has_off = any(s in question_text for s in OFF_TOPIC_SIGNALS)
        if not has_fahmai:
            if has_off:
                return RouteResult('NOT_RELATED', 0, hardcoded_answer=10, confidence='HIGH')
            choices_text = ' '.join(str(v) for v in choices.values())
            if not any(kw in choices_text for kw in FAHMAI_KEYWORDS[:20]):
                return RouteResult('NOT_RELATED', 0, hardcoded_answer=10, confidence='MEDIUM')
        if has_off and has_fahmai:
            specific = ['ฟ้าใหม่', 'FahMai', 'สายฟ้า', 'SaiFah', 'ดาวเหนือ', 'DaoNuea',
                        'คลื่นเสียง', 'KluenSiang', 'วงโคจร', 'WongKhoJon', 'จุดเชื่อม', 'JudChuam',
                        'ZenByte', 'NovaTech', 'PulseGear', 'ArcWave',
                        'AirBook', 'StormBook', 'ProBook', 'CreatorBook', 'FlexBook',
                        'HeadPro', 'HeadOn', 'SoundBar', 'Watch S3', 'Band 8', 'X9',
                        'Tab S', 'Tab A', 'DuoPad', 'Buds Z', 'Hub', 'Dock']
            if not any(kw in question_text for kw in specific):
                return RouteResult('NOT_RELATED', 0, hardcoded_answer=10, confidence='MEDIUM')

    # Stage 2: Product Identification
    found_products = product_mapper.find_products(question_text, choices)
    empty_choices = {i: '' for i in range(1, 11)}
    question_only = product_mapper.find_products(question_text, empty_choices)

    # Stage 3: Type Classification
    n_products = len(found_products)
    n_q_products = len(question_only)
    product_files = [p['filepath'] for p in found_products]

    # NOT_IN_KB
    for pattern in NOT_IN_KB_PATTERNS:
        if re.search(pattern, question_text, re.IGNORECASE):
            return RouteResult('NOT_IN_KB', 0, hardcoded_answer=9, confidence='HIGH',
                             matched_products=found_products, relevant_files=product_files)

    policy_score = sum(1 for kw in POLICY_KEYWORDS if kw in question_text)
    store_score = sum(1 for kw in STORE_KEYWORDS if kw in question_text)
    ALL_POLICIES = ['policies/cancellation_policy.md', 'policies/membership_points_policy.md',
                    'policies/return_policy.md', 'policies/shipping_policy.md', 'policies/warranty_policy.md']

    # STORE_INFO
    if store_score >= 1 and n_q_products == 0:
        return RouteResult('STORE_INFO', 3,
            relevant_files=['store_info/about_fahmai.md', 'store_info/general_faq.md',
                           'store_info/buying_guide_smartphones_2569.md'],
            confidence='HIGH', matched_products=found_products)

    # POLICY (strong, no products)
    if policy_score >= 2 and n_q_products == 0:
        return RouteResult('POLICY', 3, relevant_files=ALL_POLICIES, confidence='HIGH', matched_products=found_products)

    # POLICY about product
    if policy_score >= 1 and n_q_products >= 1:
        if re.search(r'ประกัน|เคลม|คืน|ส่งคืน|Care\+|on-site|ซ่อม|Mega Sale|จัดส่ง|ค่าส่ง|ส่งแบบด่วน|ส่ง.*ฟรี|สมาชิก.*ซื้อ|สมาชิก.*พอยท์|สมาชิก.*Points|คะแนน.*สมาชิก|Points.*ซื้อ|Pre-order|พรีออเดอร์|สั่งจอง|ยกเลิก', question_text):
            q_files = [p['filepath'] for p in question_only] if question_only else product_files
            return RouteResult('POLICY', 3, relevant_files=ALL_POLICIES + q_files,
                             confidence='HIGH', matched_products=found_products)

    # POLICY (weak, no products)
    if policy_score >= 1 and n_q_products == 0:
        return RouteResult('POLICY', 3, relevant_files=ALL_POLICIES, confidence='MEDIUM', matched_products=found_products)

    # Compare / Multi-lookup detection
    compare_kw = r'เทียบ|เปรียบเทียบ|ต่างกัน|เหมือนกัน|ตัวไหน.*ดีกว่า|อันไหน.*นานกว่า|ดีกว่า|แพงกว่า|ถูกกว่า|เร็วกว่า|หนักกว่า|เบากว่า|ทั้งคู่|ทั้งสอง|ทั้งสาม|สเปคเหมือน|จริงไหม.*สเปค|DDR\d'
    has_compare = bool(re.search(compare_kw, question_text))
    is_multi = bool(re.search(r'ราคารวม|รวมเท่าไ|พร้อมกัน.*ราคา|ซื้อ.*ทั้งหมด|รวมทั้งหมด|เซ็ต.*ราคา|ซื้อ.*กี่บาท|ราคา.*รวม|รวม.*บาท|รวมกัน', question_text))

    if n_q_products >= 2 and is_multi:
        return RouteResult('PRODUCT_SPEC', 3, relevant_files=product_files, confidence='HIGH', matched_products=found_products)

    if bool(re.search(r'รุ่นปี\s*\d{4}|รุ่น\s*\d{4}|ปี\s*\d{4}', question_text)) and n_q_products >= 2:
        return RouteResult('PRODUCT_COMPARE', 3, relevant_files=product_files, confidence='HIGH', matched_products=found_products)

    if n_q_products >= 2 and has_compare:
        return RouteResult('PRODUCT_COMPARE', 3, relevant_files=product_files, confidence='HIGH', matched_products=found_products)

    if n_q_products >= 2:
        return RouteResult('PRODUCT_SPEC', 3, relevant_files=product_files, confidence='MEDIUM', matched_products=found_products)

    if n_q_products == 1:
        q_files = [p['filepath'] for p in question_only]
        return RouteResult('PRODUCT_SPEC', 1, relevant_files=q_files, confidence='HIGH', matched_products=found_products)

    if n_q_products == 0 and n_products >= 1:
        return RouteResult('PRODUCT_SPEC', 3, relevant_files=product_files, confidence='MEDIUM', matched_products=found_products)

    if n_products == 0:
        if re.search(r'งบ|budget|ราคาไม่เกิน|แนะนำ|ตัวไหนดี|เลือกรุ่น', question_text):
            cat_files = _get_category_files(question_text, product_files)
            return RouteResult('PRODUCT_SPEC', 3, relevant_files=cat_files,
                             confidence='MEDIUM' if cat_files else 'LOW', matched_products=found_products)
        if re.search(r'โน้ตบุ๊ค|แล็ปท็อป|สมาร์ทโฟน|มือถือ|แท็บเล็ต|หูฟัง|ลำโพง|นาฬิกา', question_text):
            return RouteResult('PRODUCT_SPEC', 3, relevant_files=[], confidence='LOW', matched_products=found_products)
        return RouteResult('UNKNOWN', 3, relevant_files=[], confidence='LOW', matched_products=found_products)

    return RouteResult('UNKNOWN', 3, relevant_files=[], confidence='LOW', matched_products=found_products)

print('Question classifier พร้อมใช้งาน')

## 10. Prompt Templates — Sprint 5 Enhanced

In [ ]:
def format_choices(choices):
    return '\n'.join(f'{i}. {choices[i]}' for i in range(1, 11) if i in choices)

def product_spec_prompt(question, choices, file_content):
    """Prompt สำหรับ PRODUCT_SPEC — Sprint 5: รวมทุกแบรนด์"""
    choices_text = format_choices(choices)
    is_budget = bool(re.search(r'งบ|budget|ราคาไม่เกิน|ไม่เกิน.*บาท|ราคา.*เท่าไ|มีรุ่นไหน|กี่รุ่น|ซื้อรุ่นไหน|แนะนำ|มี.*ให้เลือก|ซื้อ.*ได้บ้าง', question))
    if is_budget:
        sys_msg = ('คุณคือผู้ช่วยร้านฟ้าใหม่ ตอบจากข้อมูลที่ให้เท่านั้น '
            'คิดทีละขั้นตอน: '
            '1) สรุปสินค้าแต่ละตัว (ชื่อ, ราคา, สเปคสำคัญ) — รวมทุกแบรนด์ทั้งแบรนด์หลักและแบรนด์พาร์ทเนอร์ '
            '2) เช็คว่าแต่ละตัวตรงตามเงื่อนไขทุกข้อหรือไม่ (ราคา, สเปค, ฟีเจอร์) '
            '3) นับจำนวนที่ตรงเงื่อนไขทุกข้อ '
            '4) เลือกคำตอบที่ตรงกับจำนวนหรือรุ่นที่ตรง '
            'สำคัญ: ต้องพิจารณาสินค้าทุกตัวในข้อมูล ห้ามข้ามแบรนด์ใดแบรนด์หนึ่ง '
            'ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน')
    else:
        sys_msg = ('คุณคือผู้ช่วยร้านฟ้าใหม่ ตอบจากข้อมูลที่ให้เท่านั้น '
            'คิดทีละขั้นตอน วิเคราะห์ข้อมูลก่อนตอบ '
            'ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน')
    return [{'role': 'system', 'content': sys_msg},
            {'role': 'user', 'content': f'ข้อมูลสินค้า:\n{file_content}\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_text}\n\nจากข้อมูลสินค้าด้านบน วิเคราะห์แต่ละตัวเลือกแล้วเลือกคำตอบที่ถูกต้อง\nตอบ ANSWER: X (เลขตัวเลือก 1-10 เท่านั้น)'}]

def product_compare_prompt(question, choices, file_contents):
    choices_text = format_choices(choices)
    combined = '\n\n---\n\n'.join(file_contents)
    return [{'role': 'system', 'content': 'คุณคือผู้ช่วยร้านฟ้าใหม่ ตอบจากข้อมูลที่ให้เท่านั้น คิดทีละขั้นตอน: 1) สรุปสเปคสำคัญของแต่ละสินค้าเป็นตาราง 2) เปรียบเทียบจุดที่คำถามถามโดยเฉพาะ 3) ตรวจสอบว่าข้อมูลตรงกับตัวเลือกไหน ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน'},
            {'role': 'user', 'content': f'ข้อมูลสินค้า:\n{combined}\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_text}\n\nเปรียบเทียบจากข้อมูลด้านบน วิเคราะห์แต่ละตัวเลือกแล้วเลือกคำตอบที่ถูกต้อง\nตอบ ANSWER: X (เลขตัวเลือก 1-10 เท่านั้น)'}]

def policy_prompt(question, choices, policy_contents, product_content=None):
    """Sprint 5: MIN cap + Clearance awareness"""
    choices_text = format_choices(choices)
    combined = '\n\n---\n\n'.join(policy_contents)
    context = f'นโยบายร้านฟ้าใหม่:\n{combined}'
    if product_content: context += f'\n\n---\n\nข้อมูลสินค้าที่เกี่ยวข้อง:\n{product_content}'
    is_calc = bool(re.search(r'กี่\s*(Points|พอยท์|บาท|วัน)|เท่าไหร่|สูงสุด|ลดได้|ค่าส่ง|ค่าจัดส่ง|จ่าย.*เท่าไ|ได้กี่|กี่\s*FahMai', question))
    if is_calc:
        sys_msg = ('คุณคือผู้เชี่ยวชาญนโยบายร้านฟ้าใหม่ ตอบจากนโยบายที่ให้เท่านั้น '
            'คำนวณทีละขั้นตอน แสดงวิธีคิดอย่างละเอียด เช่น: ราคา × อัตราพอยท์ = ?, Points ÷ อัตราแลก = ส่วนลด '
            'สำคัญมาก: ตรวจสอบเงื่อนไข สูงสุด หรือ ไม่เกิน เสมอ '
            'ถ้ามีเพดานสูงสุด ให้ใช้ค่าที่น้อยกว่าระหว่างผลคำนวณกับเพดาน: MIN(ผลคำนวณ, เพดาน) '
            'ตรวจสอบระดับสมาชิก ข้อยกเว้น ค่าขนส่ง ค่าบริการเพิ่มเติม '
            'ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน')
    else:
        sys_msg = ('คุณคือผู้เชี่ยวชาญนโยบายร้านฟ้าใหม่ ตอบจากนโยบายที่ให้เท่านั้น '
            'คิดทีละขั้นตอน อ่านนโยบายอย่างละเอียด สังเกตเงื่อนไขและข้อยกเว้น '
            'สำคัญ: อ่านข้อยกเว้นและเงื่อนไขพิเศษ เช่น สินค้าลดราคา/Clearance อาจมีนโยบายต่างจากสินค้าปกติ '
            'ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน')
    return [{'role': 'system', 'content': sys_msg},
            {'role': 'user', 'content': f'{context}\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_text}\n\nจากนโยบายด้านบน วิเคราะห์แต่ละตัวเลือกแล้วเลือกคำตอบที่ถูกต้อง\nตอบ ANSWER: X (เลขตัวเลือก 1-10 เท่านั้น)'}]

def store_info_prompt(question, choices, store_contents):
    choices_text = format_choices(choices)
    combined = '\n\n---\n\n'.join(store_contents)
    return [{'role': 'system', 'content': 'คุณคือผู้ช่วยร้านฟ้าใหม่ ตอบจากข้อมูลร้านค้าที่ให้เท่านั้น คิดทีละขั้นตอน วิเคราะห์ข้อมูลก่อนตอบ ห้ามเดาคำตอบ ต้องมีหลักฐานจากข้อมูลสนับสนุน'},
            {'role': 'user', 'content': f'ข้อมูลร้านฟ้าใหม่:\n{combined}\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_text}\n\nจากข้อมูลด้านบน วิเคราะห์แต่ละตัวเลือกแล้วเลือกคำตอบที่ถูกต้อง\nตอบ ANSWER: X (เลขตัวเลือก 1-10 เท่านั้น)'}]

def unknown_prompt(question, choices, file_contents):
    choices_text = format_choices(choices)
    combined = '\n\n---\n\n'.join(file_contents) if file_contents else '(ไม่มีข้อมูล)'
    return [{'role': 'system', 'content': 'คุณคือผู้ช่วยร้านฟ้าใหม่ ตอบจากข้อมูลที่ให้เท่านั้น คิดทีละขั้นตอน วิเคราะห์ข้อมูลก่อนตอบ ถ้าไม่มีข้อมูลเพียงพอ ให้เลือกตัวเลือกที่ 9 (ไม่มีข้อมูล) ถ้าคำถามไม่เกี่ยวกับร้าน ให้เลือกตัวเลือกที่ 10'},
            {'role': 'user', 'content': f'ข้อมูลที่เกี่ยวข้อง:\n{combined}\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_text}\n\nวิเคราะห์แต่ละตัวเลือกแล้วเลือกคำตอบที่ถูกต้อง\nตอบ ANSWER: X (เลขตัวเลือก 1-10 เท่านั้น)'}]

print('Prompt templates พร้อมใช้งาน')

## 11. Choice Shuffler — ลด Positional Bias

In [ ]:
import random
from collections import Counter

N_SHUFFLES = 3  # 3 shuffles × 3 models = 9 votes

def create_shuffled_choices(choices, seed=None):
    """สร้างตัวเลือกแบบสลับตำแหน่ง"""
    rng = random.Random(seed)
    valid_keys = sorted([k for k in choices if choices[k].strip()])
    valid_contents = [choices[k] for k in valid_keys]
    shuffled = list(valid_contents)
    rng.shuffle(shuffled)
    shuffled_choices = {}
    mapping = {}  # new_pos → original_pos
    for i, new_content in enumerate(shuffled):
        new_pos = valid_keys[i]
        for orig_key in valid_keys:
            if choices[orig_key] == new_content:
                mapping[new_pos] = orig_key
                break
        shuffled_choices[new_pos] = new_content
    return shuffled_choices, mapping

def map_answer_back(answer, mapping):
    return mapping.get(answer, answer)

def solve_with_shuffle_voting(question, original_choices, route, kb_files, model_name, query_fn, n_shuffles=N_SHUFFLES):
    """รันคำถามด้วย N_SHUFFLES ลำดับตัวเลือกที่ต่างกัน แล้ว vote"""
    answers = []
    for idx in range(n_shuffles):
        if idx == 0:
            answer = query_fn(question, original_choices, route, kb_files, model_name)
            if answer is not None: answers.append(answer)
        else:
            seed = hash(f'{question}_{model_name}_{idx}') % (2**31)
            shuffled, mapping = create_shuffled_choices(original_choices, seed=seed)
            answer = query_fn(question, shuffled, route, kb_files, model_name)
            if answer is not None:
                answers.append(map_answer_back(answer, mapping))
    if not answers: return None
    counts = Counter(answers)
    return counts.most_common(1)[0][0]

print('Choice shuffler พร้อมใช้งาน')

## 12. Strategy Full File + Main Pipeline

In [ ]:
def _sort_files_by_relevance(question, file_paths, kb_files):
    """เรียงไฟล์ตามความเกี่ยวข้องกับคำถาม"""
    if len(file_paths) <= 1: return file_paths
    q_words = set(question.lower().split())
    scored = []
    for fp in file_paths:
        content = kb_files.get(fp, '')
        first_part = content[:500].lower()
        score = sum(1 for w in q_words if w in first_part)
        scored.append((fp, score))
    scored.sort(key=lambda x: -x[1])
    return [fp for fp, _ in scored]

def build_messages(question, choices, route, kb_files):
    """สร้าง prompt messages ตามประเภทคำถาม"""
    qtype = route.question_type
    relevant_files = route.relevant_files

    if qtype == 'PRODUCT_SPEC':
        relevant_files = _sort_files_by_relevance(question, relevant_files, kb_files)
        # Budget question → summary table
        is_budget = bool(re.search(r'งบ|budget|ราคาไม่เกิน|ไม่เกิน.*บาท|มีรุ่นไหน|กี่รุ่น|ซื้อรุ่นไหน|มี.*ให้เลือก|ซื้อ.*ได้บ้าง', question))
        if is_budget and len(relevant_files) >= 3:
            summary = build_summary_table(question, kb_files, relevant_files)
            if summary:
                return product_spec_prompt(question, choices, summary)
        if relevant_files:
            if len(relevant_files) == 1:
                fp = relevant_files[0]
                if fp in kb_files:
                    contents = budget_check_and_truncate_files([kb_files[fp]])
                    return product_spec_prompt(question, choices, contents[0])
            else:
                contents = [kb_files[f] for f in relevant_files if f in kb_files]
                if contents:
                    contents = budget_check_and_truncate_files(contents)
                    return product_spec_prompt(question, choices, '\n\n---\n\n'.join(contents))
        return unknown_prompt(question, choices, [])

    elif qtype == 'PRODUCT_COMPARE':
        relevant_files = _sort_files_by_relevance(question, relevant_files, kb_files)
        selected = select_relevant_files_for_compare(question, relevant_files, kb_files, max_files=3)
        contents = [kb_files[f] for f in selected if f in kb_files]
        contents = budget_check_and_truncate_files(contents)
        return product_compare_prompt(question, choices, contents)

    elif qtype == 'POLICY':
        policy_files = [f for f in relevant_files if f.startswith('policies/')]
        product_files_p = [f for f in relevant_files if not f.startswith('policies/')]
        policy_contents, product_content = build_policy_context(question, kb_files, product_files_p)
        if not policy_contents:
            policy_contents = [kb_files[f] for f in policy_files if f in kb_files]
        policy_contents = budget_check_and_truncate_files(policy_contents)
        return policy_prompt(question, choices, policy_contents, product_content)

    elif qtype == 'STORE_INFO':
        contents = [kb_files[f] for f in relevant_files if f in kb_files]
        contents = budget_check_and_truncate_files(contents)
        return store_info_prompt(question, choices, contents)

    else:  # UNKNOWN
        contents = [kb_files[f] for f in relevant_files if f in kb_files]
        if contents: contents = budget_check_and_truncate_files(contents)
        return unknown_prompt(question, choices, contents)

def query_single_model(question, choices, route, kb_files, model_name):
    """Query โมเดลเดียว → parsed answer"""
    messages = build_messages(question, choices, route, kb_files)
    if not messages: return None
    # Budget safety net
    token_est = estimate_messages_tokens(messages)
    if token_est > MAX_CONTEXT_TOKENS:
        messages = apply_budget(messages, kb_files, route.question_type, route.relevant_files)
    response = ask_llm(messages, model=model_name, max_tokens=1024, temperature=0.1)
    if response: return parse_answer(response)
    return None

def _apply_anti_bias(votes, voting_models):
    """Sprint 5: Anti-bias + improved 3-way split tie-breaking"""
    if not votes: return None
    sorted_votes = sorted(votes.items(), key=lambda x: len(x[1]), reverse=True)
    top_answer, top_voters = sorted_votes[0]
    # 3-way split handling
    if len(sorted_votes) >= 3:
        top_count = len(top_voters)
        if all(len(v) == top_count for _, v in sorted_votes[:3]):
            for ans, _ in sorted_votes:
                if ans != 1 and ans != 9: return ans
            for ans, _ in sorted_votes:
                if ans != 1: return ans
    # Answer 1 needs strong consensus
    if top_answer == 1 and len(sorted_votes) > 1:
        second_answer, second_voters = sorted_votes[1]
        if len(second_voters) >= len(top_voters): return second_answer
        if len(top_voters) < 3 and len(second_voters) >= 1: return second_answer
    return top_answer

print('Strategy + Pipeline core พร้อมใช้งาน')

## 13. รันระบบ Pipeline

ระบบจะ:
1. จำแนกประเภทคำถาม (4-stage router)
2. สร้าง context จาก KB
3. Query 3 โมเดล × 3 shuffles = 9 votes ต่อคำถาม
4. ใช้ anti-bias voting เพื่อเลือกคำตอบ

In [ ]:
# Voting models (pathumma excluded — 50% accuracy)
VOTING_MODELS = ['typhoon', 'openthaigpt', 'kbtg']

question_list = questions[:N_QUESTIONS]
predictions = {}
start_time = time.time()

print(f'เริ่มประมวลผล {len(question_list)} คำถาม')
print(f'โมเดล: {VOTING_MODELS}')
print(f'Shuffles per model: {N_SHUFFLES}')
print(f'Total votes per question: {N_SHUFFLES * len(VOTING_MODELS)}')
print('=' * 70)

for idx, q in enumerate(question_list):
    qid = q['id']
    question_text = q['question']
    choices = q['choices']

    # จำแนกคำถาม
    route = classify_question(question_text, choices, mapper)

    # Hardcoded (NOT_RELATED=10, NOT_IN_KB=9)
    if route.hardcoded_answer is not None:
        predictions[qid] = route.hardcoded_answer
        elapsed = time.time() - start_time
        print(f'  Q{qid:3d}: [{route.question_type}] → {route.hardcoded_answer} (hardcoded) [{idx+1}/{len(question_list)}] {elapsed:.1f}s')
        continue

    # Embedding fallback ถ้าไม่มีไฟล์
    if not route.relevant_files:
        route.relevant_files = find_relevant_files_tfidf(question_text, top_k=3, kb_files=kb_files)

    # Multi-model voting with choice shuffling
    votes = {}  # answer → [model_list]
    model_answers = {}

    for model_name in VOTING_MODELS:
        # Shuffle voting: 3 permutations per model
        answer = solve_with_shuffle_voting(
            question_text, choices, route, kb_files,
            model_name, query_single_model, n_shuffles=N_SHUFFLES
        )
        model_answers[model_name] = answer
        if answer is not None:
            if answer not in votes: votes[answer] = []
            votes[answer].append(model_name)
        time.sleep(0.2)  # ป้องกัน rate limit

    # Anti-bias voting
    if votes:
        best = _apply_anti_bias(votes, VOTING_MODELS)
    else:
        best = 1  # fallback

    predictions[qid] = best
    elapsed = time.time() - start_time
    print(f'  Q{qid:3d}: [{route.question_type:15s}] → {best:2d} | votes={dict(votes)} | models={model_answers} [{idx+1}/{len(question_list)}] {elapsed:.1f}s')

total_time = time.time() - start_time
print(f'\nเสร็จสิ้น! ใช้เวลา {total_time:.1f}s | API calls: {_request_count}')

## 14. สร้างไฟล์ submission.csv

In [ ]:
# สร้าง submission.csv
output_path = '/content/submission.csv'

with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'answer'])
    for q in questions:  # ทุกคำถาม (100 ข้อ)
        qid = q['id']
        if qid in predictions:
            answer = predictions[qid]
        else:
            # คำถามที่ยังไม่ได้ run (ถ้า N_QUESTIONS < 100)
            answer = 1  # default
        writer.writerow([qid, answer])

print(f'submission.csv สร้างเรียบร้อย: {output_path}')
print(f'จำนวนคำตอบที่ประมวลผลจริง: {len(predictions)}')
print(f'จำนวนคำตอบ default (ยังไม่ได้ run): {len(questions) - len(predictions)}')

# แสดงตัวอย่าง
print('\nตัวอย่างคำตอบ:')
with open(output_path, 'r') as f:
    for i, line in enumerate(f):
        if i <= 10: print(f'  {line.strip()}')

## 15. หมายเหตุ

- เปลี่ยน `N_QUESTIONS = 100` ใน cell ที่ 1 เพื่อรัน full run
- ระบบใช้ 3 โมเดล × 3 shuffles = 9 API calls ต่อคำถาม (ยกเว้น hardcoded)
- ประมาณ 900 API calls สำหรับ 100 คำถาม (ลบ ~10 hardcoded)
- Sprint 5 improvements: accessory dedup, better prompts (budget/policy), anti-bias 3-way split